In [ ]:
import os
import cv2
import json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import argparse

classes = 'classes names'

parser = argparse.ArgumentParser()
parser.add_argument('--image_path', default='./images',type=str, help="path of images")
parser.add_argument('--label_path', default='./labels',type=str, help="path of labels .txt")
parser.add_argument('--save_path', type=str,default='./val.json', help="if not split the dataset, give a path to a json file")
arg = parser.parse_args()

def yolo2coco(arg):
    print("Loading data from ", arg.image_path, arg.label_path)

    assert os.path.exists(arg.image_path)
    assert os.path.exists(arg.label_path)
    
    originImagesDir = arg.image_path                                   
    originLabelsDir = arg.label_path
    # images dir name
    indexes = os.listdir(originImagesDir)

    dataset = {'categories': [], 'annotations': [], 'images': []}
    for i, cls in enumerate(classes, 0):
        dataset['categories'].append({'id': i, 'name': cls, 'supercategory': 'mark'})
    
    # 标注的id
    ann_id_cnt = 0
    for k, index in enumerate(tqdm(indexes)):
        # 支持 png jpg 格式的图片.
        txtFile = f'{index[:index.rfind(".")]}.txt'
        stem = index[:index.rfind(".")]
        # 读取图像的宽和高
        try:
            im = cv2.imread(os.path.join(originImagesDir, index))
            height, width, _ = im.shape
        except Exception as e:
            print(f'{os.path.join(originImagesDir, index)} read error.\nerror:{e}')
        # 添加图像的信息
        if not os.path.exists(os.path.join(originLabelsDir, txtFile)):
            # 如没标签，跳过，只保留图片信息.
            continue
        dataset['images'].append({'file_name': index,
                            'id': stem,
                            'width': width,
                            'height': height})
        with open(os.path.join(originLabelsDir, txtFile), 'r') as fr:
            labelList = fr.readlines()
            for label in labelList:
                label = label.strip().split()
                x = float(label[1])
                y = float(label[2])
                w = float(label[3])
                h = float(label[4])

                # convert x,y,w,h to x1,y1,x2,y2
                H, W, _ = im.shape
                x1 = (x - w / 2) * W
                y1 = (y - h / 2) * H
                x2 = (x + w / 2) * W
                y2 = (y + h / 2) * H
                # 标签序号从0开始计算
                cls_id = int(label[0])   
                width = max(0, x2 - x1)
                height = max(0, y2 - y1)
                dataset['annotations'].append({
                    'area': width * height,
                    'bbox': [x1, y1, width, height],
                    'category_id': cls_id,
                    'id': ann_id_cnt,
                    'image_id': stem,
                    'iscrowd': 0,
                    # mask, 矩形是从左上角点按顺时针的四个顶点
                    'segmentation': [[x1, y1, x2, y1, x2, y2, x1, y2]]
                })
                ann_id_cnt += 1

    # 保存结果
    with open(arg.save_path, 'w') as f:
        json.dump(dataset, f)
        print('Save annotation to {}'.format(arg.save_path))

if __name__ == "__main__":
    yolo2coco(arg)

In [ ]:
import os
import shutil
from tqdm import tqdm


def copy_files(source_folder, target_folder, filenames):
    for filename in tqdm(filenames, desc=f"Copying files to {target_folder}"):
        src_file = os.path.join(source_folder, filename + ".jpg")
        tgt_file = os.path.join(target_folder, filename + ".jpg")
        shutil.copy(src_file, tgt_file)


def copy_labels(source_folder, target_folder, filenames):
    for filename in tqdm(filenames, desc=f"Copying labels to {target_folder}"):
        src_txt = os.path.join(source_folder, filename + ".txt")
        tgt_txt = os.path.join(target_folder, filename + ".txt")
        shutil.copy(src_txt, tgt_txt)


def main():
    main_folder = r"Main"
    img_folder = r"img"
    label_folder = r"final_txt"
    end_folder = r"end2end"
    images_folder = r"images"
    labels_folder = r"labels"

    # 读取train.txt，val.txt和test.txt中的文件名
    train_txt = os.path.join(main_folder, "train.txt")
    val_txt = os.path.join(main_folder, "val.txt")
    test_txt = os.path.join(main_folder, "test.txt")

    with open(train_txt, "r") as f:
        train_files = f.read().strip().split("\n")

    with open(val_txt, "r") as f:
        val_files = f.read().strip().split("\n")

    with open(test_txt, "r") as f:
        test_files = f.read().strip().split("\n")

    # 创建end2end文件夹以及其子文件夹
    os.makedirs(os.path.join(end_folder, images_folder, "train"), exist_ok=True)
    os.makedirs(os.path.join(end_folder, images_folder, "val"), exist_ok=True)
    os.makedirs(os.path.join(end_folder, images_folder, "test"), exist_ok=True)
    os.makedirs(os.path.join(end_folder, labels_folder, "train"), exist_ok=True)
    os.makedirs(os.path.join(end_folder, labels_folder, "val"), exist_ok=True)
    os.makedirs(os.path.join(end_folder, labels_folder, "test"), exist_ok=True)

    # 复制图片文件到新的文件夹中
    copy_files(img_folder, os.path.join(end_folder, images_folder, "train"), train_files)
    copy_files(img_folder, os.path.join(end_folder, images_folder, "val"), val_files)
    copy_files(img_folder, os.path.join(end_folder, images_folder, "test"), test_files)

    # 复制标签txt文件到新的文件夹中
    copy_labels(label_folder, os.path.join(end_folder, labels_folder, "train"), train_files)
    copy_labels(label_folder, os.path.join(end_folder, labels_folder, "val"), val_files)
    copy_labels(label_folder, os.path.join(end_folder, labels_folder, "test"), test_files)

    print("文件复制完成！")


if __name__ == "__main__":
    main()


In [ ]:
import argparse
import glob
import os
import xml.etree.ElementTree as ET
import json
from tqdm import tqdm


def read_xml_gtbox_and_label(xml_path):
    """
        读取xml内容
    """

    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)
    depth = int(size.find('depth').text)
    points = []
    for obj in root.iter('object'):
        cls = obj.find('name').text
        pose = obj.find('pose').text
        xmlbox = obj.find('bndbox')
        xmin = float(xmlbox.find('xmin').text)
        xmax = float(xmlbox.find('xmax').text)
        ymin = float(xmlbox.find('ymin').text)
        ymax = float(xmlbox.find('ymax').text)
        box = [xmin, ymin, xmax, ymax]
        point = [cls, box]
        points.append(point)
    return points, width, height


def main():
    """
        主函数
    """
    # args = parse_args()
    raw_label_dir = r'./Annotations'
    save_dir = r'./labelme_label'
    pic_dir = r'./test'

    labels = glob.glob(raw_label_dir + '/*.xml')
    images = glob.glob(pic_dir + '/*.jpg')
    images_base_name = [os.path.basename(img).split('.')[0] for img in images]
    print(len(images_base_name))
    for i, label_abs in tqdm(enumerate(labels), total=len(labels)):
        basename = os.path.basename(label_abs).split('.')[0]
        if basename not in images_base_name:
            continue
        _, label = os.path.split(label_abs)
        label_name = label.rstrip('.xml')

        img_path = os.path.join(pic_dir, label_name + '.jpg')
        img_path = label_name + '.jpg'
        points, width, height = read_xml_gtbox_and_label(label_abs)
        json_str = {}
        json_str['version'] = '4.5.6'
        json_str['flags'] = {}
        shapes = []
        for i in range(len(points)):
            # 判断是否是左下角的点为关键点
            if points[i][0] == "left head":
                shape = {}
                shape['label'] = 'head'
                shape['points'] = [[points[i][1][0], points[i][1][3]]]
                shape['group_id'] = None
                # 类型为点
                shape['shape_type'] = 'point'
                shape['flags'] = {}
                shapes.append(shape)
            # 判断是否是右下角的点是关键点
            elif points[i][0] == "right head":
                shape = {}
                shape['label'] = 'head'
                shape['points'] = [[points[i][1][2], points[i][1][3]]]
                shape['group_id'] = None
                shape['shape_type'] = 'point'
                shape['flags'] = {}
                shapes.append(shape)
            # 其余的情况
            else:
                shape = {}
                shape['label'] = points[i][0]
                shape['points'] = [[points[i][1][0], points[i][1][1]],
                                   [points[i][1][2], points[i][1][3]]]
                shape['group_id'] = None
                # labelIMG的标注类型基本都为长方形
                shape['shape_type'] = 'rectangle'
                shape['flags'] = {}
                shapes.append(shape)
        json_str['shapes'] = shapes
        json_str['imagePath'] = img_path
        json_str['imageData'] = None
        json_str['imageHeight'] = height
        json_str['imageWidth'] = width
        with open(os.path.join(save_dir, label_name + '.json'), 'w') as f:
            json.dump(json_str, f, indent=2)


if __name__ == '__main__':
    main()
